In [14]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
import warnings

warnings.filterwarnings("ignore")


In [50]:
x="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/data/Ivoire"

In [51]:
# 1. Charger le fichier CSV

df = pd.read_excel(x+".xlsx") 

# Vérifier les valeurs manquantes
print("Valeurs manquantes par colonne :")
print(df.isna().sum())
print("\nAperçu des données :")
print(df.head())

Valeurs manquantes par colonne :
Month                            0
Food CPI                         0
FAO Food Price Index             0
FAO Meat                         0
FAO Dairy Index                  0
FAO Cereals Index                0
FAO Oils Index                   0
FAO Sugar Index                  0
UCSB CHIRPS Rainfall: avg        0
Groundnut oil ** ($/mt)          2
Palm oil ($/mt)                  2
Palm kernel oil ($/mt)           2
Soybean oil ($/mt)               2
Sunflower oil ($/mt)            32
Maize ($/mt)                     2
Sorghum ($/mt)                  54
Rice, Thai 5%  ($/mt)            2
Rice, Thai 25%  ($/mt)           7
Rice, Thai A.1 ($/mt)            2
Rice, Viet Namese 5% ($/mt)     60
Wheat, US SRW ($/mt)             9
Wheat, US HRW ($/mt)             2
Beef ** ($/kg)                   2
Chicken ** ($/kg)                2
Lamb ** ($/kg)                   2
Sugar, EU ($/kg)                 2
Sugar, world ($/kg)              2
EUR/USD               

In [52]:
# 2. Fonction pour choisir le meilleur nombre de lags

def meilleur_lag(serie, max_lags=10):
    serie = serie.dropna()
    if len(serie) < 5:  # Trop peu de points pour un modèle AR
        return 1
    aic_values = {}
    for lag in range(1, min(max_lags, len(serie)//2)):
        try:
            model = AutoReg(serie, lags=lag, old_names=False)
            result = model.fit()
            aic_values[lag] = result.aic
        except:
            continue
    return min(aic_values, key=aic_values.get) if aic_values else 1


# 3. Fonction pour remplir les NaN avec AR

def Interpol_AR_backcaster(serie):
    if serie.isna().sum() == len(serie):
        return serie
    
    # Nombre de valeurs manquantes au début
    backcast_steps = 0
    for val in serie:
        if pd.isna(val):
            backcast_steps += 1
        else:
            break
    
    # Interpolation initiale
    serie_init = serie.interpolate(method="linear", limit_direction="both")
    
    # Déterminer le meilleur lag
    lags = meilleur_lag(serie_init)
    
    # Entraîner AR
    model = AutoReg(serie_init, lags=lags, old_names=False)
    model_fit = model.fit()
    
    # Prédire toute la série existante
    prediction = model_fit.predict(start=0, end=len(serie_init)-1)
    
    # Remplacer les NaN internes
    serie_finale = serie.copy()
    serie_finale[serie_finale.isna()] = prediction[serie_finale.isna()]
    
    # 🔄 Backcasting (uniquement si on a des NaN au début)
    backcast_vals = []
    history = serie_finale.dropna().tolist()
    
    for i in range(backcast_steps):
        coeffs = model_fit.params
        lags_values = history[:lags]  # premiers points connus
        back_val = np.mean(lags_values) if lags_values else history[0]
        backcast_vals.append(back_val)
    
    # Ajouter les valeurs estimées au début
    backcast_vals = backcast_vals[::-1]  # ordre chronologique
    serie_complete = pd.Series(backcast_vals + serie_finale.tolist())
    
    return serie_complete


In [53]:
# 4. Appliquer la fonction à toutes les colonnes numériques

df_complet = df.copy()

for col in df.columns:
    if df[col].dtype in [np.float64, np.int64]:  
        print(f"🔄 Traitement de la colonne : {col}")
        
        # Tant qu'il reste des valeurs manquantes, on continue
        iteration = 0
        while df_complet[col].isna().sum() > 0:
            iteration += 1
            print(f"   ➝ Itération {iteration} : {df_complet[col].isna().sum()} valeurs manquantes")
            
            df_complet[col] = Interpol_AR_backcaster(df_complet[col])
            
            # Sécurité : éviter boucle infinie si jamais ça bloque
            if iteration > 10:
                print(f"⚠️  Trop d'itérations pour la colonne {col}, arrêt forcé.")
                break
            


🔄 Traitement de la colonne : Food CPI
🔄 Traitement de la colonne : FAO Food Price Index
🔄 Traitement de la colonne : FAO Meat
🔄 Traitement de la colonne : FAO Dairy Index
🔄 Traitement de la colonne : FAO Cereals Index
🔄 Traitement de la colonne : FAO Oils Index
🔄 Traitement de la colonne : FAO Sugar Index
🔄 Traitement de la colonne : UCSB CHIRPS Rainfall: avg
🔄 Traitement de la colonne : Groundnut oil ** ($/mt)
   ➝ Itération 1 : 2 valeurs manquantes
🔄 Traitement de la colonne : Palm oil ($/mt)
   ➝ Itération 1 : 2 valeurs manquantes
🔄 Traitement de la colonne : Palm kernel oil ($/mt)
   ➝ Itération 1 : 2 valeurs manquantes
🔄 Traitement de la colonne : Soybean oil ($/mt)
   ➝ Itération 1 : 2 valeurs manquantes
🔄 Traitement de la colonne : Sunflower oil ($/mt)
   ➝ Itération 1 : 32 valeurs manquantes
   ➝ Itération 2 : 9 valeurs manquantes
🔄 Traitement de la colonne : Maize ($/mt)
   ➝ Itération 1 : 2 valeurs manquantes
🔄 Traitement de la colonne : Sorghum ($/mt)
   ➝ Itération 1 : 54 v

In [54]:

# Vérifier si toutes les NaN ont disparu
print("\nValeurs manquantes après traitement :")
print(df_complet.isna().sum())

# 5. Sauvegarder le fichier complété
df_complet.to_excel(x+"_completes.xlsx", index=False)
print("\n✅ Fichier Excel complété enregistré :", x+"_completes.xlsx")



Valeurs manquantes après traitement :
Month                          0
Food CPI                       0
FAO Food Price Index           0
FAO Meat                       0
FAO Dairy Index                0
FAO Cereals Index              0
FAO Oils Index                 0
FAO Sugar Index                0
UCSB CHIRPS Rainfall: avg      0
Groundnut oil ** ($/mt)        0
Palm oil ($/mt)                0
Palm kernel oil ($/mt)         0
Soybean oil ($/mt)             0
Sunflower oil ($/mt)           0
Maize ($/mt)                   0
Sorghum ($/mt)                 0
Rice, Thai 5%  ($/mt)          0
Rice, Thai 25%  ($/mt)         0
Rice, Thai A.1 ($/mt)          0
Rice, Viet Namese 5% ($/mt)    0
Wheat, US SRW ($/mt)           0
Wheat, US HRW ($/mt)           0
Beef ** ($/kg)                 0
Chicken ** ($/kg)              0
Lamb ** ($/kg)                 0
Sugar, EU ($/kg)               0
Sugar, world ($/kg)            0
EUR/USD                        0
Subsidy_Index                  0
Trad